In [1]:
import sys
import os

# 현재 파일의 상위 디렉토리(SantanderCS)를 시스템 경로에 추가
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgbm
from sklearn.linear_model import LogisticRegression

from hyperopt import hp
from hyperopt import fmin, tpe, Trials

from utils import user_utils

In [2]:
# data loading
test_df = pd.read_csv("../data/test.csv")
train_df = pd.read_csv("../data/train.csv")

In [3]:
# 컬럼 삭제 ,결측치 ,log1p 처리 및 스탠다드 스케일처
train_copy_df = train_df.copy()
# X_test = test_df.iloc[:,1:]
y_target = train_copy_df.iloc[:,-1]
train_copy_df.drop(columns=['ID','TARGET'], axis=1, inplace=True)

In [4]:
train_copy_df['var3'].replace(-999999 , 2, inplace=True)

In [5]:
# 스탠다드 스케일러 적용
from sklearn.preprocessing import StandardScaler

# StandardScaler를 초기화하고 적용합니다.
scaler = StandardScaler()
train_copy_df_scaled_values = scaler.fit_transform(train_copy_df)

# 스케일링된 데이터를 새로운 데이터프레임으로 만듭니다.
train_copy_df_scaled = pd.DataFrame(train_copy_df_scaled_values, columns=train_copy_df.columns)

# 처리된 데이터프레임의 첫 5행을 출력하여 확인합니다.
print("스케일링 후 데이터프레임:")
train_copy_df_scaled.head()

스케일링 후 데이터프레임:


,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,...,saldo_medio_var29_ult3,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38
0,-0.075835,-0.788249,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.427183
1,-0.075835,0.060753,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.372038
2,-0.075835,-0.788249,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.273191
3,-0.075835,0.292298,-0.053388,0.361427,0.138158,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,-0.291398
4,-0.075835,0.446662,-0.053388,-0.213263,-0.218813,-0.038206,-0.042103,-0.013493,-0.015538,-0.033177,...,-0.005854,-0.017408,-0.011979,-0.015597,-0.016314,-0.01565,-0.012576,-0.018817,-0.019847,0.000412


In [6]:
# 테스트 데이터 스탠다드 스케일러 적용
# ID 컬럼을 제외한 모든 컬럼
X_test = test_df.iloc[:,1:]
scaler = StandardScaler()

# 테스트 데이터 스케일러 적용
test_df_scaled_values = scaler.fit_transform(X_test)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
        train_copy_df_scaled,
        y_target,
        test_size=0.2,
        random_state= 23,
        stratify=y_target
    )

In [8]:
# lgbm , logisticregression
lgbm_clf = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=5, 
)
evals = [(X_val, y_val)] #검증에 사용할 데이터셋을 지정
lgbm_clf.fit(X_train,y_train, eval_set=evals, callbacks=[lgbm.early_stopping(50)], eval_metric='logloss')

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\tj\.conda\envs\ml_dev\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 4: invalid start byte


[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14640
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 274
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039562 -> initscore=-3.189521
[LightGBM] [Info] Start training from score -3.189521
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[198]	valid_0's binary_logloss: 0.132039


,boosting_type,'gbdt'
,num_leaves,15
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [9]:
pred = lgbm_clf.predict(X_val)
pred_proba = lgbm_clf.predict_proba(X_val)[:,1]

In [ ]:
# user_utils.get_clf_eval(y_val, pred, pred_proba, model_name='lgbm_2025-11-20_ejm')

folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8508, 정확도: 0.9603, 정밀도: 0.4000, 재현율: 0.0066, F1: 0.0131
오차행렬:
[[14596     6]
 [  598     4]]


In [11]:
print(f'acc : {accuracy_score(y_val, pred)} \nroc_auc : {roc_auc_score(y_val, pred_proba)}')

acc : 0.9602736122073139 
roc_auc : 0.8507868352808358


In [12]:
lr_clf = LogisticRegression()
lr_clf.fit(X_train,y_train)
lr_preds = lr_clf.predict(X_val)
lr_preds_proba = lr_clf.predict_proba(X_val)[:,1]
print(f'acc : {accuracy_score(y_val, lr_preds):.4f} \n roc_auc {roc_auc_score(y_val, lr_preds_proba):.4f}')

acc : 0.9599 
 roc_auc 0.8051


In [ ]:
# user_utils.get_clf_eval(y_val,lr_preds,lr_preds_proba,model_name='lr_2025-11-20_ejm')

folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8051, 정확도: 0.9599, 정밀도: 0.2500, 재현율: 0.0066, F1: 0.0129
오차행렬:
[[14590    12]
 [  598     4]]


In [15]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=23
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_val)
xgb_pred_proba = xgb.predict_proba(X_val)[:,1]

print("Accuracy:", accuracy_score(y_val, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_val, xgb_pred_proba))

Accuracy: 0.9602736122073139
ROC-AUC: 0.8420545858870649


In [ ]:
# user_utils.get_clf_eval(y_val,xgb_pred,xgb_pred_proba,model_name='xgb_2025-11-21_ejm')

folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8421, 정확도: 0.9603, 정밀도: 0.4167, 재현율: 0.0083, F1: 0.0163
오차행렬:
[[14595     7]
 [  597     5]]
